In [ ]:
import os
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

def normalize_with_global(channel, gmin, gmax):
    denom = (gmax - gmin)
    if denom > 0:
        return (channel - gmin) / denom
    return np.zeros_like(channel, dtype=np.float32)

def load_ddf_slice(ddf_path, slice_idx=80):
    ddf_img = sitk.ReadImage(ddf_path)
    ddf = sitk.GetArrayFromImage(ddf_img)  # likely (Z, Y, X, 3)
    ddf_slice = ddf[:, slice_idx, :, :]    # -> (Z, X, 3) if original is (Z,Y,X,3)
    return ddf_slice.astype(np.float32)

# ---- config ----
l_list = [4000, 4500, 5000, 5500, 6000, 6500, 7000]
base_dir = "/home/xcwu"
patient = "0000"
slice_idx = 80
gamma='2.0'
# If hp_name changes with l, build it here:
# Example: ctsmoothness_l{l}_k10_mar3000_gam2.0
def make_hp_name(l):
    return f"ctsmoothness_l{l}_k10_mar3000_gam{gamma}"

# You already have return_file_path(...) in your codebase:
# ddf_path = return_file_path(os.path.join(base_dir, hp_name), patient, 'ddf.nii.gz')

# ---- pass 1: load all slices + compute global min/max per component ----
ddf_slices = {}
mins = np.array([np.inf, np.inf, np.inf], dtype=np.float64)
maxs = np.array([-np.inf, -np.inf, -np.inf], dtype=np.float64)

for l in l_list:
    hp_name = make_hp_name(l)
    run_dir = os.path.join(base_dir, hp_name)
    ddf_path = return_file_path(run_dir, patient, "ddf.nii.gz")

    ddf_slice = load_ddf_slice(ddf_path, slice_idx=slice_idx)  # (H,W,3) effectively
    ddf_slices[l] = ddf_slice

    # component-wise global min/max
    for c in range(3):
        mins[c] = min(mins[c], float(ddf_slice[..., c].min()))
        maxs[c] = max(maxs[c], float(ddf_slice[..., c].max()))

print("Global mins (dx,dy,dz):", mins)
print("Global maxs (dx,dy,dz):", maxs)

# ---- pass 2: normalize each using global ranges + plot ----
n = len(l_list)
cols = 4  # adjust
rows = int(np.ceil(n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
axes = np.array(axes).reshape(-1)

for i, l in enumerate(l_list):
    ddf_slice = ddf_slices[l]
    dx, dy, dz = ddf_slice[..., 0], ddf_slice[..., 1], ddf_slice[..., 2]

    r = normalize_with_global(dx, mins[0], maxs[0])
    g = normalize_with_global(dy, mins[1], maxs[1])
    b = normalize_with_global(dz, mins[2], maxs[2])

    rgb = np.stack([r, g, b], axis=-1)
    rgb = np.clip(rgb, 0, 1)

    ax = axes[i]
    ax.imshow(rgb)
    ax.set_title(f"l={l}  (slice={slice_idx})")
    ax.axis("off")

# turn off unused subplots
for j in range(i+1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()